In [2]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Sequence
import numpy as np

In [16]:
ATOMIC_NUMBER = {"H": 1, "C": 6}

@dataclass
class Atom:
    symbol: str
    coord: Sequence[float]

    def __post_init__(self) -> None:
        self.symbol = self.symbol.strip().capitalize()
        self.coord = np.asarray(self.coord, dtype = float)

    def __str__(self):
        return f"{self.symbol}  {self.coord[0]:.6f} {self.coord[1]:.6f} {self.coord[2]:.6f}"

    @property 
    def atomic_number(self) -> int:
        return ATOMIC_NUMBER[self.symbol]
    
    @property
    def n_electrons(self) -> int:
        return self.atomic_number

    @classmethod
    def from_string(cls, s: str) -> Atom:
        parts = s.split()
        if len(parts) != 4:
            raise ValueError(f"Expected 'Symbol X Y Z', got {s}")
        symbol = parts[0]
        try:
            coord = np.asarray(parts[1:], dtype = float)
        except ValueError as exc:
            raise ValueError(f"Invalid coordinates in atom string {s}") from exc
        return cls(symbol, coord)

    def get_distance(self, other: Atom) -> float:
        if not isinstance(other, Atom):
            raise ValueError("Distance can only be calculated between two Atom instances")
        return np.linalg.norm(self.coord - other.coord)

In [32]:
@dataclass
class Molecule:
    atoms: List[Atom]

    @classmethod
    def from_string(cls, xyz_string: str) -> Molecule:
        lines = xyz_string.strip().splitlines()
        atoms = []
        for line in lines[2:]:
            atom = Atom.from_string(line)
            atoms.append(atom)
        return cls(atoms)   

    def __str__(self): 
        return "\n".join([str(atom) for atom in self.atoms]) 
    
    @property
    def n_electrons(self):
        return sum([atom.n_electrons for atom in self.atoms])
    
    def to_string(self) -> str:
        lines = [str(len(self.atoms)), "Generated by Molecule Class"]
        for atom in self.atoms:
            line = f"{atom.symbol} {atom.coord[0]:.6f} {atom.coord[1]:.6f} {atom.coord[2]:.6f}"
            lines.append(line)
        return "\n".join(lines)

In [33]:
benzene_xyz = """12
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""


In [34]:
benzene = Molecule.from_string(benzene_xyz)
print(benzene.to_string())

12
Generated by Molecule Class
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000


In [39]:
import py3Dmol

def visualize(molecule: Molecule):
    xyz = molecule.to_string()
    view = py3Dmol.view(width = 400, height = 400)
    view.addModel(xyz, "xyz")
    view.setStyle({'stick': {}})
    view.zoomTo()
    return view


In [40]:
view = visualize(benzene)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [53]:
from dataclasses import field
import numpy as np
@dataclass
class Shell:
    l : int
    exponents: np.ndarray
    coefficiens: np.ndarray
    norm_factors: np.ndarray = field(init=False)
    center: np.ndarray =field(init = False)

In [ ]:
@dataclass
class BasisSet:
    name: str
    elements: dict[str, list[Shell]] = field(default_factory = dict)

In [7]:
from functools import lru_cache

#@lru_cache
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

fibarray = []
for i in range(40):
    fibarray.append(fib(i))
fibarray

[0,
 1,
 1,
 2,
 3,
 5,
 8,
 13,
 21,
 34,
 55,
 89,
 144,
 233,
 377,
 610,
 987,
 1597,
 2584,
 4181,
 6765,
 10946,
 17711,
 28657,
 46368,
 75025,
 121393,
 196418,
 317811,
 514229,
 832040,
 1346269,
 2178309,
 3524578,
 5702887,
 9227465,
 14930352,
 24157817,
 39088169,
 63245986]